In [ ]:
import requests
import json
import time
import os
from bs4 import BeautifulSoup
import html

DATA_DIR = "data/literature"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
def clean_text(text):
    """
    Cleans incoming text by unescaping HTML entities 
    and stripping tags.
    """
    if not text:
        return ""
    
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    
    return text.strip()

def build_query(gene, variant):
    """
    Builds the search query with specified gene and variant.
    """
    return f"(\"{gene}\" AND \"{variant}\") AND (pathogenic OR aggregation OR misfolding)"

In [ ]:
def save_corpus(gene, variant, papers):
    """
    Saves the fetched papers to a JSON corpus file.
    """
    path = f"{DATA_DIR}/{gene}_{variant}_corpus.json"
    
    if not papers:
        print("[WARNING] No literature found — saving empty file")
    
    with open(path, "w", encoding="utf-8") as f:
        json.dump(papers, f, indent=4, ensure_ascii=False)
    
    print(f"[INFO] Saved to {path}")

In [ ]:
def fetch_europepmc(query, max_papers=100):
    """
    Fetches peer-reviewed literature from Europe PMC.
    
    Filters:
    - Removes preprints (PPR)
    - Removes empty/short abstracts
    - Deduplicates by PMID
    """
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    
    papers = []
    seen = set()
    cursor = "*"
    
    while len(papers) < max_papers:
        params = {
            "query": query,
            "format": "json",
            "pageSize": 50,
            "resultType": "core",
            "cursorMark": cursor
        }
        
        # Retry with exponential backoff
        for attempt in range(3):
            try:
                time.sleep(1)  # rate limiting
                
                resp = requests.get(url, params=params, timeout=10)
                resp.raise_for_status()
                data = resp.json()
                break
            
            except Exception as e:
                print(f"[WARNING] Retry {attempt+1} failed: {e}")
                time.sleep(2 ** attempt)
        else:
            print("[ERROR] API failed after retries")
            return papers
        
        if not isinstance(data, dict):
            break
        
        results = data.get("resultList", {}).get("result", [])
        
        if not results:
            break
        
        for item in results:
            
            # Get both identifiers
            pmid = item.get("pmid")
            paper_id = item.get("id")
            
            # Remove preprints
            if paper_id and str(paper_id).startswith("PPR"):
                continue
            
            # Final ID (prefer pmid)
            pmid = pmid or paper_id
            
            # Clean text (handles HTML + entities)
            title = clean_text(item.get("title", ""))
            abstract = clean_text(item.get("abstractText", ""))
            
            # Filtering
            if not pmid:
                continue
            if not abstract or len(abstract) < 100:
                continue
            
            # Deduplication
            if pmid in seen:
                continue
            seen.add(pmid)
            
            # Store clean schema
            papers.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract
            })
            
            if len(papers) >= max_papers:
                break
        
        # Pagination
        cursor = data.get("nextCursorMark")
        if not cursor:
            break
    
    print(f"[INFO] Final valid papers: {len(papers)}")
    return papers

In [ ]:
def fetch_literature(gene, variant):
    """
    Main entry point to fetch and save literature for a specific gene and variant.
    """
    query = build_query(gene, variant)
    print(f"[INFO] Query: {query}")
    
    papers = fetch_europepmc(query)
    print(f"[INFO] Papers fetched: {len(papers)}")
    
    if papers:
        print("\n
[INFO] SAMPLE CLEAN TITLE:")
        print(papers[0]["title"])
    
    save_corpus(gene, variant, papers)
    
    return papers

In [ ]:
if __name__ == "__main__":
    fetch_literature("SNCA", "A53T")